# 1.3 dalis (b) — „TensorFlow Decision Forests“
### Kliento užklausos ketinimo klasifikavimas `„AdventureWorks DW using TensorFlow Decision Forests“ Colab stiliuje`

**Kaip veikia originalus „Colab“ (AdventureWorks DW + TF-DF):**
1. Imami lenteliniai (angl. *tabular*) duomenų saugyklos įrašai — be normalizavimo
   ir be rankinės požymių inžinerijos;
2. `pd_dataframe_to_tf_dataset()` `DataFrame` lentelę paverčia į TF duomenų rinkinį;
3. Mokomi **GradientBoostedTreesModel** ir **RandomForestModel** modeliai;
4. Vertinamas tikslumas; sprendimų medžių modeliai lenteliniams duomenims dažnai
   pranašesni už neuroninius tinklus ir reikalauja daug mažiau paruošimo.

**Mūsų atitikmuo:** tas pats ketinimų klasifikavimo uždavinys kaip ir 1.3(a),
tačiau TF-DF tekstą apdoroja automatiškai (`semantic=NATURAL_LANGUAGE`) —
rankinis TF-IDF žingsnis praleidžiamas. Modeliai mokomi lygiagrečiai
ir palyginami naudojant tą pačią kanoninę testavimo aibę.


## 1. Aplinkos paruošimas

In [ ]:
%pip install -q tensorflow tensorflow_decision_forests scikit-learn pandas matplotlib

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

SEED = 42
np.random.seed(SEED)

import tensorflow as tf
import tensorflow_decision_forests as tfdf
print('TF version:', tf.__version__, '| TF-DF version:', tfdf.__version__)

## 2. Duomenys
Tos pačios 252 (X, y) poros. Padalijimas: 80 % mokymui, 20 % validacijai;
kanoninė 6 užklausų aibė — galutiniam palyginimui. (Generatorius toks pat kaip ir `scripts/generate_dataset.py`.)

In [ ]:
# --- Deterministic dataset generator (same as scripts/generate_dataset.py) ---
# X = customer query text, y = intent label (6 classes).
DATA_SPEC = {
    'order_status': [
        ('Kur mano užsakymas {order}?', 'order'),
        ('Kokia yra užsakymo {order} būsena?', 'order'),
        ('Nematau užsakymo {order} paskyroje, ką daryti?', 'order'),
        ('Kaip sužinoti mano užsakymo {order} statusą?', 'order'),
        ('Ar užsakymas {order} jau išsiųstas?', 'order'),
        ('Mano užsakymo {order} statusas vis dar „Apmokėta“, ar tai normalu?', 'order'),
        ('Kada bus matoma užsakymo {order} sekimo informacija?', 'order'),
    ],
    'delivery': [
        ('Kiek trunka pristatymas į {city}?', 'city'),
        ('Ar pristatote siuntas į {city} savaitgaliais?', 'city'),
        ('Kokia pristatymo į {city} kaina?', 'city'),
        ('Kada mano siunta pasieks {city}?', 'city'),
        ('Ar taikomas nemokamas pristatymas perkant už {eur} eurų?', 'eur'),
        ('Kur rasti siuntos į {city} sekimo numerį?', 'city'),
        ('Ar galiu atsiimti siuntą į {city} paštomate?', 'city'),
    ],
    'returns': [
        ('Noriu grąžinti {item}, kaip tai padaryti?', 'item'),
        ('Ar galiu grąžinti prekę po {days} dienų?', 'days'),
        ('Kiek kainuoja grąžinti {item}?', 'item'),
        ('Kada atgausiu pinigus grąžinęs {item}?', 'item'),
        ('Ar galima grąžinti {item} originalioje pakuotėje?', 'item'),
        ('Kur rasiu grąžinimo formą norėdamas grąžinti {item}?', 'item'),
        ('Ar galima grąžinti {item}, jei pakuotė jau buvo atidaryta?', 'item'),
    ],
    'payment': [
        ('Ar priimate {method}?', 'method'),
        ('Kiek laiko trunka mokėjimo autorizacija pasirinkus {method}?', 'method'),
        ('Nepavyko apmokėti užsakymo pasirinkus {method}, ką daryti?', 'method'),
        ('Ar galima mokėti dalimis pasirinkus {method}?', 'method'),
        ('Kada gausiu sąskaitą faktūrą apmokėjus per {method}?', 'method'),
        ('Ar palaikote {method} internetinėje parduotuvėje?', 'method'),
        ('Ką daryti, jei mokant per {method} nurašė pinigus, bet užsakymas neapmokėtas?', 'method'),
    ],
    'warranty': [
        ('Kiek laiko galioja garantija įsigijus {item}?', 'item'),
        ('Ar nemokamai suremontuosite {item} pagal garantiją?', 'item'),
        ('Ką daryti, jei gaminys sugedo po {days} dienų naudojimo?', 'days'),
        ('Ar reikia atskiro talono pateikiant {item} garantiniam remontui?', 'item'),
        ('Kiek trunka garantinis remontas pateikus {item}?', 'item'),
        ('Ar garantija galioja, jei netyčia pažeidžiau {item}?', 'item'),
        ('Ar atvežate {item} į remontą nemokamai?', 'item'),
    ],
    'product_usage': [
        ('Kaip susieti {item} su telefonu?', 'item'),
        ('Kur rasti naudotojo vadovą įsigijus {item}?', 'item'),
        ('Nusipirkau {item}, bet prietaisas neįsijungia, ką daryti?', 'item'),
        ('Kaip atstatyti gamyklinius nustatymus naudojant {item}?', 'item'),
        ('Ar galima gauti naudotojo vadovą lietuvių kalba įsigijus {item}?', 'item'),
        ('Kaip tinkamai prijungti {item} prie kompiuterio?', 'item'),
        ('Kiek laiko reikia krauti {item}?', 'item'),
    ],
}

SLOT_VALUES = {
    'order': ['TS-102345', 'TS-201111', 'TS-314560', 'TS-456789', 'TS-511235', 'TS-623456'],
    'city': ['Vilnių', 'Kauną', 'Klaipėdą', 'Šiaulius', 'Panevėžį', 'Alytų'],
    'item': ['išmanųjį telefoną', 'belaides ausines', 'išorinę bateriją',
             'elektroninę skaityklę', 'žaidimų pultelį', 'internetinę kamerą'],
    'days': ['3', '7', '10', '13', '20', '30'],
    'eur': ['30', '45', '60', '75', '90', '120'],
    'method': ['PayPal', 'Visa kortelę', 'Mastercard kortelę',
               'bankinį pavedimą', 'Google Pay', 'Apple Pay'],
}

INTENTS = ['order_status', 'delivery', 'returns', 'payment', 'warranty', 'product_usage']


def generate_dataset():
    rows = []
    for intent, specs in DATA_SPEC.items():
        for template, slot in specs:
            for value in SLOT_VALUES[slot]:
                rows.append((template.replace('{' + slot + '}', value), intent))
    return pd.DataFrame(rows, columns=['query', 'intent'])


df = generate_dataset()
print(f'Dataset: {len(df)} (X, y) pairs, {len(INTENTS)} classes')
print(df['intent'].value_counts())

# Canonical hold-out test set (same 6 queries as data/customer_support_test.csv
# and as the 1.2.2 Gemini prototype) - used to compare all models fairly.
TEST_EXAMPLES = [
    ('Kokiomis dienomis pristatote užsakymus į Vilnių?', 'delivery'),
    ('Norėčiau grąžinti prekę po 10 dienų nuo pirkimo, ar įmanoma?', 'returns'),
    ('Nematau savo užsakymo istorijos paskyroje, ką daryti?', 'order_status'),
    ('Ar parduotuvė priima PayPal mokėjimus?', 'payment'),
    ('Ar už 25 eurus pirktas gaminys turi garantiją?', 'warranty'),
    ('Kur rasti ausinių susiejimo instrukciją?', 'product_usage'),
]

# 80 / 20 split for training and validation
train_df = df.sample(frac=0.8, random_state=SEED)
val_df = df.drop(train_df.index)

train_ds = tfdf.keras.pd_dataframe_to_tf_dataset(train_df, label='intent')
val_ds = tfdf.keras.pd_dataframe_to_tf_dataset(val_df, label='intent')
print(f'Train: {len(train_df)} rows, Validation: {len(val_df)} rows')

# TF-DF sorts class labels alphabetically - we must use the same order for predictions
CLASSES = sorted(df['intent'].unique())
print('TF-DF class order:', CLASSES)

## 3. Kaip mokosi TF-DF modeliai
- **RandomForestModel** (*bagging*): mokomos dešimtys medžių, kiekvienas mato
  atsitiktinį požymių ir pavyzdžių poaibį; galutinis sprendimas — balsavimas.
  Triukšmo (angl. *noise*) atžvilgiu labai stabilus.
- **GradientBoostedTreesModel** (*boosting*): medžiai mokomi nuosekliai,
  kiekvienas taiso ankstesniųjų klaidas; dažnai tikslesnis su mažais duomenų rinkiniais,
  bet jautresnis persimokymui.
Teksto stulpelį `query` TF-DF atpažįsta automatiškai (`NATURAL_LANGUAGE` semantika).

In [ ]:
rf_model = tfdf.keras.RandomForestModel(num_trees=50, random_seed=SEED)
rf_model.compile(metrics=['accuracy'])
rf_model.fit(train_ds, validation_data=val_ds, verbose=0)

rf_eval = rf_model.evaluate(val_ds, return_dict=True)
print(f"RandomForest validation accuracy: {rf_eval['accuracy']:.4f}")

In [ ]:
gbt_model = tfdf.keras.GradientBoostedTreesModel(random_seed=SEED)
gbt_model.compile(metrics=['accuracy'])
gbt_model.fit(train_ds, validation_data=val_ds, verbose=0)

gbt_eval = gbt_model.evaluate(val_ds, return_dict=True)
print(f"GradientBoostedTrees validation accuracy: {gbt_eval['accuracy']:.4f}")

## 4. Modelių palyginimas
Vertiname abu modelius: (a) validacijos aibėje, (b) kanoninėje 6 užklausų
testavimo aibėje. Painiavos matrica parodo, kurios klasės supainiojamos.

Painiavos matricos braižymo funkcija tokia pati kaip ir 1.3(a) dalyje.

In [ ]:
def plot_confusion_matrix(y_true, y_pred, classes, title):
    cm = np.zeros((len(classes), len(classes)), dtype=int)
    for t, p in zip(y_true, y_pred):
        cm[t, p] += 1
    fig, ax = plt.subplots(figsize=(5.5, 5))
    im = ax.imshow(cm, cmap='Blues')
    ax.set_xticks(range(len(classes)), classes, rotation=45, ha='right')
    ax.set_yticks(range(len(classes)), classes)
    for i in range(len(classes)):
        for j in range(len(classes)):
            ax.text(j, i, cm[i, j], ha='center', va='center',
                    color='white' if cm[i, j] > cm.max() / 2 else 'black')
    ax.set_xlabel('Numatytas ketinimas')
    ax.set_ylabel('Tikrasis ketinimas')
    ax.set_title(title)
    fig.colorbar(im)
    plt.tight_layout()
    plt.show()
    return cm

from sklearn.metrics import accuracy_score

# predictions need feature-only datasets (no label column)
val_features_ds = tfdf.keras.pd_dataframe_to_tf_dataset(val_df[['query']])
test_df = pd.DataFrame({'query': [q for q, _ in TEST_EXAMPLES]})
test_ds = tfdf.keras.pd_dataframe_to_tf_dataset(test_df)

y_val_true = [CLASSES.index(l) for l in val_df['intent']]
y_val_pred_rf = np.argmax(rf_model.predict(val_features_ds, verbose=0), axis=1)

test_true = [CLASSES.index(l) for _, l in TEST_EXAMPLES]
test_pred_rf = np.argmax(rf_model.predict(test_ds, verbose=0), axis=1)
test_pred_gbt = np.argmax(gbt_model.predict(test_ds, verbose=0), axis=1)

rf_test_acc = accuracy_score(test_true, test_pred_rf)
gbt_test_acc = accuracy_score(test_true, test_pred_gbt)

print(f'Canonical test accuracy - RandomForest: {rf_test_acc:.2%} | GBT: {gbt_test_acc:.2%}')
print()
for (query, true_label), pred_id in zip(TEST_EXAMPLES, test_pred_rf):
    mark = 'OK ' if CLASSES[pred_id] == true_label else 'WRONG'
    print(f'{mark} true={true_label:14s} pred={CLASSES[pred_id]:14s} {query}')

# --- bar chart: validation vs test accuracy for both models ---
fig, ax = plt.subplots(figsize=(7, 4))
models = ['RandomForest', 'GradientBoostedTrees']
val_scores = [rf_eval['accuracy'], gbt_eval['accuracy']]
test_scores = [rf_test_acc, gbt_test_acc]
x = np.arange(len(models))
ax.bar(x - 0.18, val_scores, 0.36, label='Validacija', color='#2563eb')
ax.bar(x + 0.18, test_scores, 0.36, label='Testas (6 užklausos)', color='#4c9f70')
ax.set_xticks(x, models)
ax.set_ylim(0, 1.05)
ax.set_ylabel('Tikslumas')
ax.set_title('TF-DF modelių palyginimas')
ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

plot_confusion_matrix(y_val_true, y_val_pred_rf, CLASSES,
                      'RandomForest — validacijos painiavos matrica')

## 5. Kintamųjų svarba
TF-DF `inspector` parodo, kuriuos kintamuosius modelis naudojo dažniausiai —
mūsų atveju turime vienintelį požymį (užklausos tekstą), tačiau originalioje
„AdventureWorks Colab“ užrašinėje taip matoma, kad lenteliniuose duomenyse svarbiausi
yra, pvz., kliento demografiniai ar pardavimo požymiai.

In [ ]:
inspector = rf_model.make_inspector()
print('Model type:', inspector.model_type())
print('Number of trees:', inspector.num_trees())
try:
    importances = inspector.variable_importances()['INV_MEAN_MIN_DEPTH']
    for name, value in sorted(importances, key=lambda kv: kv[1])[:5]:
        print(f'  {name}: min-depth score {value:.3f}')
except Exception as e:
    print('Variable importances not available for text features:', e)

## 6. Išvados
1. TF-DF išsprendė tą patį klasifikavimo uždavinį **be rankinės požymių
   inžinerijos** — tekstą apdorojo vidinis NLP semantinis sluoksnis.
2. Palyginus su 1.3(a) MLP: abu metodai pasiekia panašų tikslumą, bet TF-DF
   reikalavo mažiau paruošimo žingsnių ir mokėsi greičiau.
3. Sprendimų medžių modeliams nereikia normalizavimo ir jie natūraliai priima tekstą —
   dėl to jie yra geras atskaitos (angl. *baseline*) modelis tekstiniams uždaviniams su nedideliu duomenų kiekiu.
